### 지도 학습 지시 미세 튜닝을 위해 데이터셋 준비하기

In [3]:
import json

with open("instruction-data.json", "r") as file:
    data = json.load(file)
print(f"샘플 개수: {len(data)}")

샘플 개수: 1100


In [4]:
# 코드 7-2 프롬프트 포맷팅 함수 구현하기
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text + input_text

In [5]:
# 코드 7-3 데이터셋 분할하기
train_portion = int(len(data) * 0.85)                   # 935
test_portion = int(len(data) * 0.1)                     # 110
val_portion = len(data) - train_portion - test_portion  # 55

train_data = data[:train_portion]                               # [:935]
test_data = data[train_portion:train_portion + test_portion]    # [935:1045]
val_data = data[train_portion+test_portion:]                    # [1045:]

print("훈련 세트 크기: ", len(train_data))
print("검증 세트 크기: ", len(val_data))
print("테스트 세트 크기: ", len(test_data))

훈련 세트 크기:  935
검증 세트 크기:  55
테스트 세트 크기:  110


### 훈련 배치 만들기

In [6]:
# 코드 7-4 지시 데이터셋 클래스 구현하기
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [7]:
# 토크나이저의 .encode 메서드를 사용해 <|endoftext|> 토큰의 ID를 확인해보자.
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})
print(encoded) # [50256]

[50256]


In [8]:
# 사용자 정의 콜레이트 함수로 패딩 처리를 구현
def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (new_item + [pad_token_id] * (batch_max_length - len(new_item)))
        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [9]:
# custom_collate_draft_1 함수가 의도한 대로 동작하는지 테스트
# 서로 다른 세 가지 입력을 전달하여 동일한 길이로 패딩한 다음에 배치로 만들어 보자.

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (inputs_1, inputs_2, inputs_3)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [10]:
# 입력 토큰 ID에서 타깃 토큰 ID를 생성하는 콜레이트 함수
def custom_collate_draft_2(batch, pad_token_id=50256, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (new_item + [pad_token_id] * (batch_max_length - len(new_item)))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)


tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


In [11]:
# 코드 7-5 사용자 정의 콜레이트 함수 구현하기
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (new_item + [pad_token_id] * (batch_max_length - len(new_item)))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [12]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [13]:
logits_1 = torch.tensor([
    [-1.0, 1.0],
    [-0.5, 1.5]
])
print(logits_1.shape)

targets_1 = torch.tensor([0, 1])
print(targets_1.shape)

loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

torch.Size([2, 2])
torch.Size([2])
tensor(1.1269)


In [14]:
logits_2 = torch.tensor([
    [-1.0, 1.0],
    [-0.5, 1.5],
    [-0.5, 1.5],
])
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(0.7936)


In [ ]:
targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
